# Phase 3: Building Search Indices (Colab Version)

This notebook runs in Google Colab to handle large datasets.

**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Build BM25 + FAISS indices
4. Download indices to local machine

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q rank-bm25 sentence-transformers faiss-cpu pyarrow

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
import json
import time
import re
from pathlib import Path
from tqdm import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

print("✓ Imports successful")

## 2. Load Data

In [ ]:
# Load products from Google Drive
# UPDATE THIS PATH if your folder name is different
drive_path = '/content/drive/MyDrive/wholesale-project'

df_products = pd.read_parquet(f'{drive_path}/products.parquet')

print(f"✓ Loaded {len(df_products):,} products")
print(f"Columns: {list(df_products.columns)}")

In [ ]:
# Check RAM available
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Available RAM: {ram_gb:.1f} GB")

# Check if GPU is available
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU (using CPU)")

## 3. Create Searchable Text

In [ ]:
def create_product_text(row):
    """Combine product fields into searchable text."""
    parts = []
    
    if pd.notna(row.get('product_title')):
        parts.append(str(row['product_title']))
    
    if pd.notna(row.get('product_brand')):
        parts.append(f"Brand: {row['product_brand']}")
    
    if pd.notna(row.get('product_class')):
        parts.append(f"Category: {row['product_class']}")
    
    if pd.notna(row.get('product_description')):
        desc = str(row['product_description'])[:500]
        parts.append(desc)
    
    if pd.notna(row.get('product_bullet_point')):
        bullets = str(row['product_bullet_point'])[:300]
        parts.append(bullets)
    
    return " | ".join(parts)

print("Creating searchable text...")
df_products['search_text'] = df_products.apply(create_product_text, axis=1)
print(f"✓ Created search text for {len(df_products):,} products")

## 4. Build BM25 Index

In [ ]:
def tokenize(text):
    """Simple tokenizer."""
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

print("Building BM25 index...")
start_time = time.time()

corpus = df_products['search_text'].tolist()
tokenized_corpus = [tokenize(doc) for doc in tqdm(corpus, desc="Tokenizing")]

bm25 = BM25Okapi(tokenized_corpus)

print(f"✓ BM25 index built in {time.time() - start_time:.1f}s")
print(f"  Documents: {len(tokenized_corpus):,}")

## 5. Generate Embeddings

In [ ]:
# Load embedding model
print("Loading embedding model...")
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(model_name)

# Use GPU if available
if torch.cuda.is_available():
    embedding_model = embedding_model.to('cuda')
    print("✓ Model loaded on GPU")
else:
    print("✓ Model loaded on CPU")

print(f"  Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

In [ ]:
# Generate embeddings
print("\nGenerating embeddings...")
print("(This may take 5-15 minutes depending on dataset size)")

start_time = time.time()

texts_to_embed = df_products['product_title'].fillna('').tolist()

embeddings = embedding_model.encode(
    texts_to_embed,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✓ Embeddings generated in {time.time() - start_time:.1f}s")
print(f"  Shape: {embeddings.shape}")

## 6. Build FAISS Index

In [ ]:
print("Building FAISS index...")
start_time = time.time()

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)

# Create index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"✓ FAISS index built in {time.time() - start_time:.1f}s")
print(f"  Vectors: {index.ntotal:,}")
print(f"  Dimension: {dimension}")

## 7. Test Search

In [ ]:
def bm25_search(query, top_k=3):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(df_products.iloc[i]['product_title'], scores[i]) for i in top_indices]

def faiss_search(query, top_k=3):
    query_emb = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    scores, indices = index.search(query_emb, top_k)
    return [(df_products.iloc[i]['product_title'], scores[0][j]) for j, i in enumerate(indices[0])]

# Test
test_query = "ceramic mugs bulk"
print(f"Query: '{test_query}'\n")

print("BM25 Results:")
for title, score in bm25_search(test_query):
    print(f"  - {title[:60]}... ({score:.2f})")

print("\nFAISS Results:")
for title, score in faiss_search(test_query):
    print(f"  - {title[:60]}... ({score:.3f})")

## 8. Save Indices to Google Drive

In [ ]:
# Create indices folder in Google Drive
indices_dir = Path(f'{drive_path}/indices')
indices_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving to: {indices_dir}")

In [ ]:
# Save BM25
print("Saving BM25 index...")
bm25_data = {'bm25': bm25, 'tokenized_corpus': tokenized_corpus}
with open(indices_dir / 'bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25_data, f)
print("✓ BM25 saved")

# Save FAISS
print("Saving FAISS index...")
faiss.write_index(index, str(indices_dir / 'faiss_index.bin'))
print("✓ FAISS saved")

# Save embeddings
print("Saving embeddings...")
np.save(indices_dir / 'product_embeddings.npy', embeddings)
print("✓ Embeddings saved")

# Save product IDs
print("Saving product IDs...")
product_ids = df_products['product_id'].tolist()
with open(indices_dir / 'product_ids.json', 'w') as f:
    json.dump(product_ids, f)
print("✓ Product IDs saved")

## 9. Summary

In [ ]:
print("\n" + "=" * 60)
print("PHASE 3 COMPLETE (Colab)")
print("=" * 60)

print(f"""
Indices Built:
  - Products: {len(df_products):,}
  - BM25: {len(tokenized_corpus):,} documents
  - FAISS: {index.ntotal:,} vectors ({dimension}D)

Files saved to Google Drive:
  {indices_dir}/
    - bm25_index.pkl
    - faiss_index.bin
    - product_embeddings.npy
    - product_ids.json

NEXT STEPS:
1. Go to Google Drive
2. Download the 'indices' folder
3. Put it in your local: data/indices/
4. Continue with Phase 4 locally!
""")